# Model Comparison

### Requirements:

- **Lab 3 (due Mon Apr 20):** model comparison (≥3 families) + structured error analysis (failure-mode table) + one-page non-technical memo. Start thinking about your comparison strategy.


- This is the deliverable pattern. For every model you build from now on — Lab 3, Capstone, Final Exam — you will produce this table.

| Failure Mode | N (races) | F1 (slice) | F1 (overall) | Gap | Proposed Mitigation |
|---|---|---|---|---|---|
| Street circuits | ? | ? | ? | ? | ? |
| Wet races | ? | ? | ? | ? | ? |
| New circuits | ? | ? | ? | ? | ? |

> **⏸ Pause.** "This table turns a model evaluation into a robustness report. Lab 3 requires exactly this. Your Capstone requires exactly this. Learn the pattern now."


- REASONING > METHODOLOGY 	A correct comparison table with no explanation of WHY each model performed the way it did scores lower than a table with fewer models but clear reasoning. Write reasoning in markdown cells. Explain your thinking in code comments.

- AUTHENTICITY > FORMAT 	PROMPTS.md is scored for traceability (can I trace your claims to your notebook?), operational detail (exact errors, specific metrics), and critical distance. Format compliance alone scores low.

"My model lost to the baseline" is a valid, high-scoring result — as long as you explain WHY it lost and what that tells you about the problem.

- ≥3 models + baseline compared. Same metric, same validation, same test set across ALL rows. Visible cell outputs in notebook. Train-vs-test gap reported for each model.

- Key rule: Every row in your comparison table must use the same primary metric and the same temporal split. If they don't match, the comparison is invalid regardless of how many models you trained. The metric must be appropriate for your chosen framing (e.g., MAE for regression, macro F1 for classification).

- New requirement: Include a Train [metric] column alongside your test metric. The gap between train and test performance is diagnostic — it tells you whether your model is overfitting.

- Notebook runs top-to-bottom in <10 min on fresh env. RANDOM_SEED = 414 throughout. Visible outputs for every cell. README with clear instructions. Environment specified.

## 0. Setup

### Library, environment setup

In [1]:
# ── Reproducibility Header ────────────────────────────────────────────
# Every notebook in IIT414W starts here. Do not skip this block.

import sys, random
import numpy as np
import warnings

RANDOM_SEED = 414  # Course constant. Do not change.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore', category=FutureWarning)

# Environment check
print(f'Python  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'Seed    : {RANDOM_SEED}')

# ── Dependency Guard ───────────────────────────────────────────────
# Ensures all required packages are installed in the active kernel.
# Safe to re-run: pip will skip already-installed packages.

import importlib, subprocess, sys

_REQUIRED = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'fastf1': 'fastf1',
}

_missing = []
for _mod, _pip in _REQUIRED.items():
    try:
        importlib.import_module(_mod)
    except ModuleNotFoundError:
        _missing.append(_pip)

if _missing:
    print(f'Installing missing packages: {_missing}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + _missing)
    print('Done. Packages installed successfully.')
else:
    print('All required packages already installed ✓')

# ── Library Imports ───────────────────────────────────────────────
import os                        # Working directory checks
import subprocess                # Git command checks
import importlib                 # Runtime dependency checks
import numpy as np               # Numeric support
import pandas as pd              # Tables and diagnostics
import matplotlib.pyplot as plt  # Plotting
import seaborn as sns            # Statistical plotting
import fastf1                    # Formula 1 data access

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier  # Tree-based ensemble

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    classification_report, brier_score_loss,
)

try:
    from xgboost import XGBClassifier
    print("XGBoost available ✓")
except ImportError:
    XGBClassifier = None
    print("⚠ XGBoost not installed. Run: pip install xgboost")


print(f'fastf1  : {fastf1.__version__}')
print(f'pandas  : {pd.__version__}')

Python  : 3.12.3
NumPy   : 2.4.2
Seed    : 414
All required packages already installed ✓
XGBoost available ✓
fastf1  : 3.8.1
pandas  : 2.3.3


### Dataset, DataFrame setup

In [2]:
# Configure local cache so repeated API reads are fast and reproducible.
cache_path = os.path.abspath("fastf1_cache")
os.makedirs(cache_path, exist_ok=True)
fastf1.Cache.enable_cache(cache_path)

# Create Ergast interface and iterate all race weekends in each season.
erg = fastf1.ergast.Ergast()

years = [2018, 2019, 2020, 2021, 2022, 2023, 2024]
results_list = []

for year in years:
    schedule = fastf1.get_event_schedule(year)

    for _, event in schedule.iterrows():
        # Ignore testing sessions because they are not part of race prediction.
        if event['EventFormat'] == 'testing':
            continue

        rnd = int(event['RoundNumber'])

        try:
            res = erg.get_race_results(season=year, round=rnd)
            race = res.content[0]

            # Keep temporal/event metadata for later grouping and split logic.
            race['year'] = year
            race['round'] = rnd
            race['circuit'] = event['EventName']

            results_list.append(race)

        except Exception as e:
            print(f"Skipping {year} round {rnd}")

# Merge all race result frames into one modeling table.
df = pd.concat(results_list, ignore_index=True)

# Keep only columns used in this lab.
df = df[[
    'driverNumber',
    'givenName',
    'familyName',
    'constructorName',
    'grid',
    'position',
    'status',
    'points',
    'year',
    'round',
    'circuit'
]]

# Standardize column names for readability.
df.columns = [
    'DriverNumber',
    'FirstName',
    'LastName',
    'TeamName',
    'GridPosition',
    'Position',
    'Status',
    'Points',
    'Year',
    'Round',
    'Circuit'
]

# Build convenient identity and numeric fields.
df['FullName'] = df['FirstName'] + " " + df['LastName']
df['GridPosition'] = pd.to_numeric(df['GridPosition'], errors='coerce')
df['Position'] = pd.to_numeric(df['Position'], errors='coerce')

# Feature engineering for baseline and analysis.
df['Top10'] = ((df['Position'] <= 10) & (df['Position'] >0)).astype(int)
df['BaselinePred'] = (df['GridPosition'] <= 10).astype(int)
df['PositionsGained'] = df['GridPosition'] - df['Position']
df['DNF'] = ~df['Status'].str.contains("Finished", na=False)

# Historical reliability and constructor strength proxies.
driver_reliability = 1 - df.groupby('FullName')['DNF'].mean()
df['DriverReliability'] = df['FullName'].map(driver_reliability)
constructor_points = df.groupby('TeamName')['Points'].mean()
df['ConstructorStrength'] = df['TeamName'].map(constructor_points)

# Leakage-safe versions: each season uses only information from prior seasons.
driver_year = (
    df.groupby(['FullName', 'Year'], as_index=False)
      .agg(SeasonStarts=('DNF', 'size'), SeasonDNFs=('DNF', 'sum'))
      .sort_values(['FullName', 'Year'])
)
driver_year['CumStarts'] = driver_year.groupby('FullName')['SeasonStarts'].cumsum()
driver_year['CumDNFs'] = driver_year.groupby('FullName')['SeasonDNFs'].cumsum()
driver_year['PastStarts'] = driver_year.groupby('FullName')['CumStarts'].shift(1).fillna(0)
driver_year['PastDNFs'] = driver_year.groupby('FullName')['CumDNFs'].shift(1).fillna(0)
driver_year['DriverReliabilityPast'] = np.where(
    driver_year['PastStarts'] > 0,
    1 - (driver_year['PastDNFs'] / driver_year['PastStarts']),
    np.nan,
)

team_year = (
    df.groupby(['TeamName', 'Year'], as_index=False)
      .agg(SeasonStarts=('Points', 'size'), SeasonPoints=('Points', 'sum'))
      .sort_values(['TeamName', 'Year'])
)
team_year['CumStarts'] = team_year.groupby('TeamName')['SeasonStarts'].cumsum()
team_year['CumPoints'] = team_year.groupby('TeamName')['SeasonPoints'].cumsum()
team_year['PastStarts'] = team_year.groupby('TeamName')['CumStarts'].shift(1).fillna(0)
team_year['PastPoints'] = team_year.groupby('TeamName')['CumPoints'].shift(1).fillna(0)
team_year['ConstructorStrengthPast'] = np.where(
    team_year['PastStarts'] > 0,
    team_year['PastPoints'] / team_year['PastStarts'],
    np.nan,
)

df = df.merge(
    driver_year[['FullName', 'Year', 'DriverReliabilityPast']],
    on=['FullName', 'Year'],
    how='left',
)
df = df.merge(
    team_year[['TeamName', 'Year', 'ConstructorStrengthPast']],
    on=['TeamName', 'Year'],
    how='left',
)

# Remove unusable rows for grid/finish-based analyses.
df = df.dropna(subset=['GridPosition','Position'])

print("Dataset shape:", df.shape)


Request for URL https://api.jolpi.ca/ergast/f1/2022/11/results.json failed; using cached response
Traceback (most recent call last):
  File "/home/hacker-m/Desktop/UDD/2026/Trimestre_01/AI_Workshop/venv/lib/python3.12/site-packages/requests_cache/session.py", line 297, in _resend
    response.raise_for_status()
  File "/home/hacker-m/Desktop/UDD/2026/Trimestre_01/AI_Workshop/venv/lib/python3.12/site-packages/requests/models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 429 Client Error: Too Many Requests for url: https://api.jolpi.ca/ergast/f1/2022/11/results.json
Request for URL https://api.jolpi.ca/ergast/f1/2022/12/results.json failed; using cached response
Traceback (most recent call last):
  File "/home/hacker-m/Desktop/UDD/2026/Trimestre_01/AI_Workshop/venv/lib/python3.12/site-packages/requests_cache/session.py", line 297, in _resend
    response.raise_for_status()
  File "/home/hacker-m/Desktop/UDD/2026/Trim

Dataset shape: (2979, 20)


In [3]:
# Quick sanity check of available seasons before splitting
print(df["Year"].value_counts().sort_index())

Year
2018    420
2019    420
2020    340
2021    440
2022    440
2023    440
2024    479
Name: count, dtype: int64


In [4]:
# Temporal split: train (<2022), validation (2022), test (>=2023)
train = df[df["Year"] <= 2021].copy()
val   = df[df["Year"] == 2022].copy()
test  = df[df["Year"] >= 2023].copy()

known_circuits = set(train["Circuit"].dropna())
df["CircuitNovelty"] = np.where(df["Circuit"].isin(known_circuits), "known", "new")

train = df[df["Year"] <= 2021].copy()
val   = df[df["Year"] == 2022].copy()
test  = df[df["Year"] >= 2023].copy()

In [5]:

# Features: keep slicing context out of the model.
CONTEXT_COLUMNS = ["Year", "Round", "Circuit", "CircuitNovelty", "TeamName", "FullName"]
NUMERIC_FEATURES = [
    "GridPosition", "Round", "Year", "DriverReliabilityPast", "ConstructorStrengthPast",
]
CATEGORICAL_FEATURES = ["Circuit", "CircuitNovelty", "TeamName", "FullName"]
TARGET = "DNF"

available_num = [c for c in NUMERIC_FEATURES if c in train.columns]
available_cat = [c for c in CATEGORICAL_FEATURES if c in train.columns]
MODEL_FEATURES = available_num + available_cat

def make_preprocessor():
    transformers = []
    if available_num:
        transformers.append(("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), available_num))
    if available_cat:
        transformers.append(("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), available_cat))
    return ColumnTransformer(transformers)

>Our main target is going to be "DNF"

## Model 1: Logistic Regression

**Why?:** The target is binary ("finished" vs "did not finish"), so Logistic Regression is a simple baseline to compare against more complex models.

In [6]:
def make_lr_pipeline():
    return Pipeline([
        ("prep", make_preprocessor()),
        ("clf", LogisticRegression(random_state=RANDOM_SEED, max_iter=1000)),
    ])



## Model 2: XGB Classifier

**Why?:** It handles complex feature relationships and works well for classification with non-linear patterns.

In [7]:
def make_xgb_pipeline():
    if XGBClassifier is None:
        raise ImportError("XGBoost is required for this notebook. Install with: pip install xgboost")
    return Pipeline([
        ("prep", make_preprocessor()),
        ("clf", XGBClassifier(
            n_estimators=150, learning_rate=0.03, max_depth=4,
            subsample=0.9, colsample_bytree=0.9,
            random_state=RANDOM_SEED, eval_metric="logloss", n_jobs=1,
        )),
    ])

## Model 3: Random Forest Classifier

**Why?:**  It requires minimal tuning to get running and we have enough data to avoid memorization (4 years worth of data)

In [8]:
preprocessor_rf = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', NUMERIC_FEATURES),   # No scaling needed for trees
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES),
    ],
    remainder='drop'
)

pipe_rf = Pipeline([
    ('prep', preprocessor_rf),
    ('model', RandomForestClassifier(
        n_estimators=200,
        max_depth=10,            # Limit depth to reduce overfitting on small data
        min_samples_leaf=5,      # Require at least 5 samples per leaf
        random_state=RANDOM_SEED,
        n_jobs=-1
    ))
])

## Comparison Table

### 1. Calling the 3 models (train, test)

In [9]:
X_train = train[MODEL_FEATURES]
y_train = train[TARGET]
X_val = val[MODEL_FEATURES]
y_val = val[TARGET]
X_test = test[MODEL_FEATURES]
y_test = test[TARGET]

lr_pipe = make_lr_pipeline()
rf_pipe = pipe_rf
xgb_pipe = make_xgb_pipeline()
lr_pipe.fit(X_train, y_train)
rf_pipe.fit(X_train, y_train)
xgb_pipe.fit(X_train, y_train)

y_pred_lr_val = lr_pipe.predict(X_val)
y_pred_rf_val = rf_pipe.predict(X_val)
y_pred_xgb_val = xgb_pipe.predict(X_val)

y_pred_lr = lr_pipe.predict(X_test)
y_pred_rf = rf_pipe.predict(X_test)
y_pred_xgb = xgb_pipe.predict(X_test)
y_proba_rf = rf_pipe.predict_proba(X_test)[:, 1]
y_proba_xgb = xgb_pipe.predict_proba(X_test)[:, 1]

val_f1_lr = f1_score(y_val, y_pred_lr_val, zero_division=0)
val_f1_rf = f1_score(y_val, y_pred_rf_val, zero_division=0)
val_f1_xgb = f1_score(y_val, y_pred_xgb_val, zero_division=0)

overall_f1_lr = f1_score(y_test, y_pred_lr, zero_division=0)
overall_f1_rf = f1_score(y_test, y_pred_rf, zero_division=0)
overall_f1 = f1_score(y_test, y_pred_xgb, zero_division=0)

results_df = test[[
    "Year", "Round", "Circuit", "CircuitNovelty",
    "TeamName", "FullName", "GridPosition",
]].copy()
results_df["y_true"] = y_test.to_numpy()
results_df["y_pred_lr"] = y_pred_lr
results_df["y_pred_rf"] = y_pred_rf
results_df["y_pred_xgb"] = y_pred_xgb
results_df["y_proba_rf"] = y_proba_rf
results_df["y_proba_xgb"] = y_proba_xgb

line = "─" * 60
print(line)
print("Data source: Ergast race results + FastF1 schedule metadata")
print(f"Train: {len(train)} rows (≤2021)  |  Val: {len(val)} rows (2022)  |  Test: {len(test)} rows (≥2023)")
print(f"Model features: {len(available_num)} numeric + {len(available_cat)} categorical")
print(line)
print(f"Logistic F1 -> Val: {val_f1_lr:.4f} | Test: {overall_f1_lr:.4f}")
print(f"Random Forest F1 -> Val: {val_f1_rf:.4f} | Test: {overall_f1_rf:.4f}")
print(f"XGBoost F1 -> Val: {val_f1_xgb:.4f} | Test: {overall_f1:.4f}")
print(line)
display(results_df.head(10))

────────────────────────────────────────────────────────────
Data source: Ergast race results + FastF1 schedule metadata
Train: 1620 rows (≤2021)  |  Val: 440 rows (2022)  |  Test: 919 rows (≥2023)
Model features: 5 numeric + 4 categorical
────────────────────────────────────────────────────────────
Logistic F1 -> Val: 0.5500 | Test: 0.5661
Random Forest F1 -> Val: 0.5926 | Test: 0.6576
XGBoost F1 -> Val: 0.6030 | Test: 0.6553
────────────────────────────────────────────────────────────


,Year,Round,Circuit,CircuitNovelty,TeamName,FullName,GridPosition,y_true,y_pred_lr,y_pred_rf,y_pred_xgb,y_proba_rf,y_proba_xgb
2060,2023,1,Bahrain Grand Prix,known,Red Bull,Max Verstappen,1,False,False,False,0,0.188432,0.140645
2061,2023,1,Bahrain Grand Prix,known,Red Bull,Sergio Pérez,2,False,False,False,0,0.220754,0.151662
2062,2023,1,Bahrain Grand Prix,known,Aston Martin,Fernando Alonso,5,False,False,False,0,0.486513,0.376180
2063,2023,1,Bahrain Grand Prix,known,Ferrari,Carlos Sainz,4,False,False,False,0,0.208203,0.190222
2064,2023,1,Bahrain Grand Prix,known,Mercedes,Lewis Hamilton,7,False,False,False,0,0.083500,0.092121
2065,2023,1,Bahrain Grand Prix,known,Aston Martin,Lance Stroll,8,False,False,True,0,0.523278,0.469323
2066,2023,1,Bahrain Grand Prix,known,Mercedes,George Russell,6,False,False,False,0,0.242431,0.159801
2067,2023,1,Bahrain Grand Prix,known,Alfa Romeo,Valtteri Bottas,12,False,True,True,1,0.577181,0.679411
2068,2023,1,Bahrain Grand Prix,known,Alpine F1 Team,Pierre Gasly,20,False,True,False,1,0.491682,0.631560
2069,2023,1,Bahrain Grand Prix,known,Williams,Alexander Albon,15,False,True,True,1,0.707902,0.816331


### 2. Slices

In [10]:
def slice_metrics(df, slice_col, y_true_col="y_true", y_pred_col="y_pred_xgb"):
    """Compute F1, precision, recall, and sample count per slice."""
    rows = []
    for name, group in df.groupby(slice_col, dropna=False):
        if len(group) < 2:
            continue
        if len(group) < 5:
            warnings.warn(f"⚠ Slice '{name}' has only {len(group)} samples — metric may be unreliable")
        rows.append({
            "Slice": f"{slice_col} = {name}",
            "N": len(group),
            "F1": f1_score(group[y_true_col], group[y_pred_col], zero_division=0),
            "Precision": precision_score(group[y_true_col], group[y_pred_col], zero_division=0),
            "Recall": recall_score(group[y_true_col], group[y_pred_col], zero_division=0),
        })
    return pd.DataFrame(rows)


def mitigation_for_slice(slice_name):
    """Suggest domain-informed mitigations for a failing slice."""
    text = slice_name.lower()
    if "street" in text or "circuit_type" in text:
        return "Add circuit-geometry features such as track length, corner count, and average wall proximity."
    if "wet" in text or "true" in text:
        return "Flag wet-race predictions as low-confidence and require strategist review before pit-stop use."
    if "new" in text:
        return "Use a novelty flag and back-test against similar circuit layouts before deployment."
    if "post-2022" in text or "regulation" in text:
        return "Compare all-history vs post-2022 retraining and document the sample-size trade-off."
    return "Audit the slice drivers and add one domain feature that directly represents this context."


def make_failure_mode_table(results_df, y_pred_col, overall_f1, model_name, slice_col_list=None):
    """
    Generate failure-mode table for a specific model.
    
    Args:
        results_df: DataFrame with y_true and model predictions
        y_pred_col: Column name for model predictions (e.g., 'y_pred_lr', 'y_pred_rf', 'y_pred_xgb')
        overall_f1: Overall F1 score for the model on full test set
        model_name: Display name for the model (e.g., 'Logistic Regression')
        slice_col_list: List of columns to slice on. Defaults to ["Circuit", "CircuitNovelty", "TeamName"]
    
    Returns:
        DataFrame with failure modes, sample counts, slice/overall F1, gap %, and mitigations
    """
    if slice_col_list is None:
        slice_col_list = ["Circuit", "CircuitNovelty", "TeamName"]
    
    slice_tables = [
        slice_metrics(results_df, col, y_pred_col=y_pred_col)
        for col in slice_col_list
    ]
    slice_rows = pd.concat(slice_tables, ignore_index=True).sort_values("F1", ascending=True).reset_index(drop=True)
    
    worst_three = slice_rows.sort_values("F1", ascending=True).head(3).copy()
    failure_mode_table = pd.DataFrame({
        "Model": model_name,
        "Failure Mode": worst_three["Slice"].str.replace(" = ", ": ", regex=False),
        "N (races)": worst_three["N"].astype(int),
        "F1 (slice)": worst_three["F1"],
        "F1 (overall)": overall_f1,
        "Gap (%)": ((overall_f1 - worst_three["F1"]) / max(overall_f1, 1e-9) * 100).clip(lower=0),
        "Proposed Mitigation": [mitigation_for_slice(s) for s in worst_three["Slice"]],
    })
    
    return failure_mode_table


# Generate failure-mode tables for each model
print("\n" + "═" * 100)
print("FAILURE-MODE ANALYSIS: 3 Models")
print("═" * 100 + "\n")

fm_lr = make_failure_mode_table(results_df, "y_pred_lr", overall_f1_lr, "Logistic Regression")
print(f"▼ {fm_lr['Model'].iloc[0]}")
display(fm_lr[["Failure Mode", "N (races)", "F1 (slice)", "F1 (overall)", "Gap (%)", "Proposed Mitigation"]].round({"F1 (slice)": 3, "F1 (overall)": 3, "Gap (%)": 1}))

fm_rf = make_failure_mode_table(results_df, "y_pred_rf", overall_f1_rf, "Random Forest")
print(f"\n▼ {fm_rf['Model'].iloc[0]}")
display(fm_rf[["Failure Mode", "N (races)", "F1 (slice)", "F1 (overall)", "Gap (%)", "Proposed Mitigation"]].round({"F1 (slice)": 3, "F1 (overall)": 3, "Gap (%)": 1}))

fm_xgb = make_failure_mode_table(results_df, "y_pred_xgb", overall_f1, "XGBoost")
print(f"\n▼ {fm_xgb['Model'].iloc[0]}")
display(fm_xgb[["Failure Mode", "N (races)", "F1 (slice)", "F1 (overall)", "Gap (%)", "Proposed Mitigation"]].round({"F1 (slice)": 3, "F1 (overall)": 3, "Gap (%)": 1}))


════════════════════════════════════════════════════════════════════════════════════════════════════
FAILURE-MODE ANALYSIS: 3 Models
════════════════════════════════════════════════════════════════════════════════════════════════════

▼ Logistic Regression


,Failure Mode,N (races),F1 (slice),F1 (overall),Gap (%),Proposed Mitigation
0,Circuit: British Grand Prix,40,0.0,0.566,100.0,Audit the slice drivers and add one domain fea...
1,Circuit: Belgian Grand Prix,40,0.0,0.566,100.0,Audit the slice drivers and add one domain fea...
2,Circuit: Chinese Grand Prix,20,0.0,0.566,100.0,Audit the slice drivers and add one domain fea...



▼ Random Forest


,Failure Mode,N (races),F1 (slice),F1 (overall),Gap (%),Proposed Mitigation
0,TeamName: Ferrari,92,0.0,0.658,100.0,Audit the slice drivers and add one domain fea...
1,TeamName: Mercedes,92,0.0,0.658,100.0,Audit the slice drivers and add one domain fea...
2,TeamName: Red Bull,92,0.0,0.658,100.0,Audit the slice drivers and add one domain fea...



▼ XGBoost


,Failure Mode,N (races),F1 (slice),F1 (overall),Gap (%),Proposed Mitigation
0,TeamName: Red Bull,92,0.000,0.655,100.0,Audit the slice drivers and add one domain fea...
1,TeamName: Mercedes,92,0.000,0.655,100.0,Audit the slice drivers and add one domain fea...
2,Circuit: Belgian Grand Prix,40,0.133,0.655,79.7,Audit the slice drivers and add one domain fea...


### 3. Comparison Table

In [ ]:
# 3. Comparison Table: 3 Models + Baseline (Always DNF)
print("\n" + "═" * 120)
print("MODEL COMPARISON TABLE: DNF Prediction")
print("═" * 120 + "\n")

# Compute train metrics for each model
y_pred_lr_train = lr_pipe.predict(X_train)
y_pred_rf_train = rf_pipe.predict(X_train)
y_pred_xgb_train = xgb_pipe.predict(X_train)

# Baseline prediction: always predicts DNF (class = 1)
y_pred_base_always_dnf_train = np.ones(len(y_train), dtype=int)
y_pred_base_always_dnf_val = np.ones(len(y_val), dtype=int)
y_pred_base_always_dnf_test = np.ones(len(y_test), dtype=int)

train_f1_lr = f1_score(y_train, y_pred_lr_train, zero_division=0)
train_f1_rf = f1_score(y_train, y_pred_rf_train, zero_division=0)
train_f1_xgb = f1_score(y_train, y_pred_xgb_train, zero_division=0)
train_f1_base = f1_score(y_train, y_pred_base_always_dnf_train, zero_division=0)

train_precision_lr = precision_score(y_train, y_pred_lr_train, zero_division=0)
train_precision_rf = precision_score(y_train, y_pred_rf_train, zero_division=0)
train_precision_xgb = precision_score(y_train, y_pred_xgb_train, zero_division=0)
train_precision_base = precision_score(y_train, y_pred_base_always_dnf_train, zero_division=0)

train_recall_lr = recall_score(y_train, y_pred_lr_train, zero_division=0)
train_recall_rf = recall_score(y_train, y_pred_rf_train, zero_division=0)
train_recall_xgb = recall_score(y_train, y_pred_xgb_train, zero_division=0)
train_recall_base = recall_score(y_train, y_pred_base_always_dnf_train, zero_division=0)

# Build comparison table: 3 trained models + baseline
comparison_rows = [
    {
        'Model': 'Baseline: Always DNF',
        'n_features': 0,
        'Train F1': train_f1_base,
        'Val F1': f1_score(y_val, y_pred_base_always_dnf_val, zero_division=0),
        'Test F1': f1_score(y_test, y_pred_base_always_dnf_test, zero_division=0),
        'F1 Gap': f1_score(y_test, y_pred_base_always_dnf_test, zero_division=0) - train_f1_base,

    },
    {
        'Model': 'Logistic Regression',
        'n_features': len(available_num) + len(available_cat),
        'Train F1': train_f1_lr,
        'Val F1': val_f1_lr,
        'Test F1': overall_f1_lr,
        'F1 Gap': overall_f1_lr - train_f1_lr,

    },
    {
        'Model': 'Random Forest',
        'n_features': len(available_num) + len(available_cat),
        'Train F1': train_f1_rf,
        'Val F1': val_f1_rf,
        'Test F1': overall_f1_rf,
        'F1 Gap': overall_f1_rf - train_f1_rf,

    },
    {
        'Model': 'XGBoost',
        'n_features': len(available_num) + len(available_cat),
        'Train F1': train_f1_xgb,
        'Val F1': val_f1_xgb,
        'Test F1': overall_f1,
        'F1 Gap': overall_f1 - train_f1_xgb,

    },
]

comparison_df = pd.DataFrame(comparison_rows)

# Round for readability
comparison_df = comparison_df.round(3)

print("Full metric breakdown:")
display(comparison_df)

# Summary: pick best model by test F1
best_model_idx = comparison_df['Test F1'].idxmax()
best_model = comparison_df.loc[best_model_idx]

print(f"\n{'─' * 120}")
print(f"BEST MODEL (by Test F1): {best_model['Model']}")
print(f"  Test F1: {best_model['Test F1']:.4f}  |  Val F1: {best_model['Val F1']:.4f}  |  Train F1: {best_model['Train F1']:.4f}")

if best_model['F1 Gap'] > 0.05:
    print(f"  ⚠ Overfitting signal: large positive gap ({best_model['F1 Gap']:.4f}) suggests high variance.")
elif best_model['F1 Gap'] < -0.05:
    print(f"  ⚠ Underfitting signal: negative gap ({best_model['F1 Gap']:.4f}) suggests high bias.")
else:
    print(f"  ✓ Generalization looks healthy (gap ≈ 0).")
print(f"{'─' * 120}")



════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
MODEL COMPARISON TABLE: DNF Prediction
════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

Full metric breakdown:


,Model,n_features,Train F1,Val F1,Test F1,F1 Gap
0,Baseline: Always DNF,0,0.675,0.536,0.526,-0.149
1,Logistic Regression,9,0.776,0.550,0.566,-0.210
2,Random Forest,9,0.772,0.593,0.658,-0.114
3,XGBoost,9,0.798,0.603,0.655,-0.143



────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
BEST MODEL (by Test F1): Random Forest
  Test F1: 0.6580  |  Val F1: 0.5930  |  Train F1: 0.7720
  ⚠ Underfitting signal: negative gap (-0.1140) suggests high bias.
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
